# Natural Selection, Adaptation, and Fitness Workflow

This notebook scaffold supports the article **Natural Selection, Adaptation, and Fitness**. It can be expanded with genotype-specific selection, mean fitness, allele-frequency updates, quantitative trait response, variable-environment selection, time-series screening, condition scoring, and provenance documentation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

article_dir = Path.cwd().parent
scenarios = pd.read_csv(article_dir / 'data' / 'selection_scenarios.csv')
p = scenarios['p0']
q = 1 - p
scenarios['mean_fitness'] = p**2 * scenarios['w_AA'] + 2 * p * q * scenarios['w_Aa'] + q**2 * scenarios['w_aa']
scenarios['p_next'] = (p**2 * scenarios['w_AA'] + p * q * scenarios['w_Aa']) / scenarios['mean_fitness']
scenarios[['scenario', 'p0', 'p_next', 'mean_fitness']].round(4)

In [ ]:
traits = pd.read_csv(article_dir / 'data' / 'trait_observations.csv')
traits['relative_fitness'] = np.exp(0.6 * traits['trait_value'])
traits['relative_fitness'] = traits['relative_fitness'] / traits['relative_fitness'].mean()
mean_before = traits['trait_value'].mean()
selected_mean = np.average(traits['trait_value'], weights=traits['relative_fitness'])
S = selected_mean - mean_before
h2 = 0.45
R = h2 * S
pd.DataFrame({'mean_before':[mean_before], 'selected_mean':[selected_mean], 'S':[S], 'h2':[h2], 'R':[R]}).round(4)

In [ ]:
ts = pd.read_csv(article_dir / 'data' / 'allele_frequency_timeseries.csv')
ts['delta_p'] = ts['allele_frequency'].diff()
ts['delta_p_per_time'] = ts['delta_p'] / ts['time'].diff()
ts.round(4)

In [ ]:
condition = pd.read_csv(article_dir / 'data' / 'selection_condition_sites.csv')
condition['selection_condition_score'] = (
    0.18 * condition['standing_variation'] +
    0.18 * condition['selection_strength'] +
    0.18 * condition['environmental_match'] +
    0.16 * condition['demographic_stability'] +
    0.14 * condition['gene_flow_support'] +
    0.16 * (1 - condition['constraint_risk'])
)
condition.sort_values('selection_condition_score', ascending=False).round(3)